# Solution to p20

Directed mean aggregation uses receiver rows exactly as supplied: `A @ X / degree` with degree summed over the sender axis. The four projected aggregate tokens are preceded by a zero class token. Self-attention uses that same five-token tensor as Q, K, and V, then LayerNorm is applied to the residual before token 0 reaches the classifier. Actual features and aligned targets are fingerprint-validated on every loss call.

In [ ]:
import importlib.util
import random
import sys
from pathlib import Path

import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

SEED=20260901
ATOL=1e-5
RTOL=1e-5
TRAIN_IDS=('graph-train-00','graph-train-01','graph-train-02','graph-train-03')
HELDOUT_IDS=('graph-heldout-00','graph-heldout-01')

fixture_path=Path("units/B2-021-cross-modal-transformers-vision/data/vision_fixture.py")
fixture_spec=importlib.util.spec_from_file_location("p20_vision_fixture",fixture_path)
vision_fixture=importlib.util.module_from_spec(fixture_spec)
sys.modules[fixture_spec.name]=vision_fixture
fixture_spec.loader.exec_module(vision_fixture)

def seed_everything():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    torch.set_num_threads(1); torch.use_deterministic_algorithms(True)

def _to_torch(fields): return {name:torch.from_numpy(value.copy()) for name,value in fields.items()}
def _to_numpy(fields):
    converted={}
    for name,value in fields.items():
        if not isinstance(value,torch.Tensor): raise TypeError("batch fields must be tensors")
        converted[name]=value.detach().cpu().contiguous().numpy().copy()
    return converted

def build_train_batch(ids):
    if tuple(ids)!=TRAIN_IDS: raise ValueError("train IDs must equal immutable stored order")
    features,targets=vision_fixture.build_task_batch('graph', tuple(ids))
    return _to_torch(features),_to_torch(targets)

def build_heldout_batch(ids):
    if tuple(ids)!=HELDOUT_IDS: raise ValueError("held-out IDs must equal immutable stored order")
    features,targets=vision_fixture.build_task_batch('graph', tuple(ids))
    return _to_torch(features),_to_torch(targets)

def validate_actual(features,targets):
    role,ids=vision_fixture.validate_actual_batch('graph',_to_numpy(features),_to_numpy(targets))
    nodes,adjacency,labels=features.get("node_features"),features.get("adjacency"),targets.get("label")
    if nodes is None or adjacency is None or labels is None: raise ValueError("missing graph fields")
    B=nodes.shape[0]
    if nodes.shape!=(B,4,3) or adjacency.shape!=(B,4,4) or labels.shape!=(B,): raise ValueError("graph shape mismatch")
    if nodes.dtype!=torch.float32 or adjacency.dtype!=torch.uint8 or labels.dtype!=torch.int64: raise ValueError("dtype mismatch")
    if not bool(torch.all((adjacency==0)|(adjacency==1))): raise ValueError("adjacency must be binary")
    if bool(torch.any(adjacency.sum(dim=2)==0)): raise ValueError("receiver degree must be positive")
    return role,ids

def mean_neighbor_aggregate(nodes,adjacency):
    adjacency_float=adjacency.to(torch.float32)
    degree=adjacency_float.sum(dim=2,keepdim=True)
    if bool(torch.any(degree==0)): raise ValueError("zero-degree receiver")
    return torch.bmm(adjacency_float,nodes)/degree

class GraphTokenClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.projection=nn.Linear(3,8)
        self.class_token=nn.Parameter(torch.zeros(1,1,8))
        self.attention=nn.MultiheadAttention(8,1,dropout=0,batch_first=True)
        self.norm=nn.LayerNorm(8,eps=1e-5)
        self.head=nn.Linear(8,2)
    def forward(self,nodes,adjacency,return_aux=False):
        aggregated=mean_neighbor_aggregate(nodes,adjacency)
        projected=self.projection(aggregated)
        expanded_class=self.class_token.expand(nodes.shape[0],-1,-1)
        attention_input=torch.cat((expanded_class,projected),dim=1)
        attended,_=self.attention(attention_input,attention_input,attention_input,need_weights=False)
        normalized=self.norm(attention_input+attended)
        logits=self.head(normalized[:,0,:])
        if return_aux: return logits,{"aggregated":aggregated,"attention_input":attention_input,"normalized_class":normalized[:,0:1,:]}
        return logits

def compute_loss(model,features,targets):
    validate_actual(features,targets)
    logits=model(features["node_features"],features["adjacency"])
    if logits.shape!=(features["node_features"].shape[0],2): raise ValueError("logit shape mismatch")
    return F.cross_entropy(logits,targets["label"],reduction="mean")

def train_graph_token_classifier(model,batch,optimizer):
    features,targets=batch
    role,ids=validate_actual(features,targets)
    if role!="train" or ids!=TRAIN_IDS: raise ValueError("optimizer batch must be train only")
    trace=[]; model.train()
    for update in range(1,13):
        optimizer.zero_grad(set_to_none=True)
        loss=compute_loss(model,features,targets)
        loss.backward(); optimizer.step()
        trace.append({"update":update,"mean_ce":float(loss.detach())})
    return tuple(trace)

train_batch=build_train_batch(TRAIN_IDS)
heldout_batch=build_heldout_batch(HELDOUT_IDS)
seed_everything()
model=GraphTokenClassifier()
optimizer=torch.optim.AdamW(model.parameters(),lr=.05,betas=(.9,.999),eps=1e-8,weight_decay=0,amsgrad=False,foreach=False,fused=False)
model.eval()
with torch.no_grad(): heldout_loss_before=float(compute_loss(model,*heldout_batch))
training_trace=train_graph_token_classifier(model,train_batch,optimizer)
model.eval()
with torch.no_grad():
    heldout_loss_after=float(compute_loss(model,*heldout_batch))
    heldout_logits,heldout_aux=model(heldout_batch[0]["node_features"],heldout_batch[0]["adjacency"],return_aux=True)
    heldout_correct=int((heldout_logits.argmax(1)==heldout_batch[1]["label"]).sum())
certificate={"updates":len(training_trace),"heldout_improvement":heldout_loss_before-heldout_loss_after,"heldout_correct":heldout_correct,"aggregation_shape":tuple(heldout_aux["aggregated"].shape),"attention_input_shape":tuple(heldout_aux["attention_input"].shape)}


### Answer check

In [ ]:
assert len(training_trace)==12
assert tuple(row["update"] for row in training_trace)==tuple(range(1,13))
assert certificate["aggregation_shape"]==(2,4,3)
assert certificate["attention_input_shape"]==(2,5,8)
assert certificate["heldout_improvement"]>=.02
assert certificate["heldout_correct"]>=2
assert set(TRAIN_IDS).isdisjoint(HELDOUT_IDS)
for bad in (TRAIN_IDS[::-1],TRAIN_IDS[:-1],(TRAIN_IDS[0],TRAIN_IDS[0],*TRAIN_IDS[2:]),(*TRAIN_IDS[:-1],'unknown-id'),HELDOUT_IDS):
    try: build_train_batch(bad)
    except ValueError: pass
    else: raise AssertionError("train wrapper accepted invalid IDs")
for bad in (HELDOUT_IDS[::-1],HELDOUT_IDS[:1],(HELDOUT_IDS[0],HELDOUT_IDS[0]),('unknown-id',HELDOUT_IDS[1]),TRAIN_IDS):
    try: build_heldout_batch(bad)
    except ValueError: pass
    else: raise AssertionError("held-out wrapper accepted invalid IDs")
assert torch.isfinite(compute_loss(model,*train_batch))
